# 01 — Data Exploration

Quick inspection of the QM9 dataset: shapes, feature dimensions, target distributions, and geometry sanity checks.

**Run from the repo root** (or let the first cell add `src/` to the path).

Prerequisites: `python scripts/preprocess.py --config configs/preprocess_qm9.yaml`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'IG-MPNN' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repo root: {ROOT}')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.data import QM9Dataset, make_splits, get_loaders
from src.data.datasets import QM9_TARGETS

## 1. Load dataset

In [ ]:
ds = QM9Dataset(
    root=ROOT / 'data',
    target_idx=0,          # dipole moment — change to explore others
    use_angles=True,
    use_torsions=True,
)

print(ds)
print(f'  Total molecules : {len(ds):,}')
print(f'  Node feature dim: {ds.num_node_features}')
print(f'  Edge feature dim: {ds.num_edge_features}')

## 2. Inspect a single graph

In [ ]:
sample = ds[0]
print(sample)
print()

attrs = [
    ('x',              'Node features',      sample.x.shape),
    ('edge_index',     'Edge index',          sample.edge_index.shape),
    ('edge_attr',      'Bond features',       sample.edge_attr.shape if sample.edge_attr is not None else 'N/A'),
    ('pos',            '3D coordinates',      sample.pos.shape),
    ('y',              'Target (y)',           sample.y.shape),
    ('edge_attr_geo',  'Bond lengths (Å)',    sample.edge_attr_geo.shape if hasattr(sample, 'edge_attr_geo') else 'missing'),
    ('angle_index',    'Angle triplet index', sample.angle_index.shape if hasattr(sample, 'angle_index') else 'missing'),
    ('angle_attr',     'Bond angles (rad)',   sample.angle_attr.shape if hasattr(sample, 'angle_attr') else 'missing'),
    ('torsion_index',  'Torsion quad index',  sample.torsion_index.shape if hasattr(sample, 'torsion_index') else 'missing'),
    ('torsion_attr',   'Torsion angles (rad)',sample.torsion_attr.shape if hasattr(sample, 'torsion_attr') else 'missing'),
]

print(f'{"Attribute":<18} {"Description":<26} {"Shape"}')
print('-' * 60)
for attr, desc, shape in attrs:
    print(f'{attr:<18} {desc:<26} {shape}')

## 3. Node feature breakdown

In [ ]:
# Feature layout matches featurizer.py
feature_names = [
    'Z / 9 (norm. atomic num)',
    'is H',
    'is C',
    'is N',
    'is O',
    'is F',
    'formal charge',
    'implicit Hs / 4',
    'in ring',
    'aromatic',
    'degree / 4',
]

x = sample.x
print(f'Shape: {x.shape}  ({x.shape[0]} atoms × {x.shape[1]} features)')
print()
print(f'{"#":<4} {"Feature":<30} {"Min":>6} {"Max":>6} {"Mean":>8}')
print('-' * 58)
for i, name in enumerate(feature_names):
    col = x[:, i]
    print(f'{i:<4} {name:<30} {col.min().item():>6.3f} {col.max().item():>6.3f} {col.mean().item():>8.4f}')

## 4. Bond feature breakdown

In [ ]:
bond_feature_names = ['single', 'double', 'triple', 'aromatic']
ea = sample.edge_attr
print(f'Shape: {ea.shape}  ({ea.shape[0]} directed edges × {ea.shape[1]} features)')
print()
for i, name in enumerate(bond_feature_names):
    count = int(ea[:, i].sum().item())
    print(f'  {name:<12}: {count} edges')

## 5. Geometry sanity checks

In [ ]:
import math

# Collect geometry stats from the first 500 molecules
N_SAMPLE = 500
all_lengths, all_angles, all_torsions = [], [], []

for i in range(N_SAMPLE):
    d = ds[i]
    if hasattr(d, 'edge_attr_geo'):
        all_lengths.append(d.edge_attr_geo.squeeze(-1))
    if hasattr(d, 'angle_attr'):
        all_angles.append(d.angle_attr.squeeze(-1))
    if hasattr(d, 'torsion_attr'):
        all_torsions.append(d.torsion_attr.squeeze(-1))

lengths  = torch.cat(all_lengths).numpy()
angles   = torch.cat(all_angles).numpy()
torsions = torch.cat(all_torsions).numpy()

for label, arr, unit in [
    ('Bond lengths',   lengths,  'Å'),
    ('Bond angles',    np.degrees(angles),   '°'),
    ('Torsion angles', np.degrees(torsions), '°'),
]:
    print(f'{label:20}  min={arr.min():.2f} {unit}  '
          f'max={arr.max():.2f} {unit}  '
          f'mean={arr.mean():.2f} {unit}  '
          f'n={len(arr):,}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(lengths, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Bond Lengths')
axes[0].set_xlabel('Distance (Å)')
axes[0].set_ylabel('Count')

axes[1].hist(np.degrees(angles), bins=60, color='seagreen', edgecolor='white', linewidth=0.3)
axes[1].set_title('Bond Angles')
axes[1].set_xlabel('Angle (°)')

axes[2].hist(np.degrees(torsions), bins=60, color='coral', edgecolor='white', linewidth=0.3)
axes[2].set_title('Torsion Angles')
axes[2].set_xlabel('Angle (°)')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(f'Geometry distributions (first {N_SAMPLE:,} molecules)', y=1.02)
plt.tight_layout()
plt.show()

## 6. Target property distributions

In [ ]:
# Load raw QM9 to get all 12 targets at once (without narrowing to one column)
from torch_geometric.datasets import QM9 as _QM9

raw_qm9 = _QM9(root=str(ROOT / 'data' / 'raw' / 'qm9'))

# Stack all targets: (N, 19) → we use the first 12
all_y = torch.stack([raw_qm9[i].y.squeeze(0) for i in range(len(raw_qm9))], dim=0)  # (N, 19)
all_y = all_y[:, :12]  # drop the rarely-used individual components

print(f'Target tensor shape: {all_y.shape}  (N molecules × 12 targets)')

In [ ]:
print(f'{"Idx":<5} {"Name":<8} {"Unit":<12} {"Min":>12} {"Max":>12} {"Mean":>12} {"Std":>12}')
print('-' * 75)
for idx, meta in QM9_TARGETS.items():
    col = all_y[:, idx]
    print(
        f'{idx:<5} {meta["name"]:<8} {meta["unit"]:<12} '
        f'{col.min().item():>12.4f} {col.max().item():>12.4f} '
        f'{col.mean().item():>12.4f} {col.std().item():>12.4f}'
    )

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for idx, meta in QM9_TARGETS.items():
    ax = axes[idx // 4][idx % 4]
    col = all_y[:, idx].numpy()
    ax.hist(col, bins=80, color='steelblue', edgecolor='none')
    ax.set_title(f"{meta['name']} [{meta['unit']}]", fontsize=9)
    ax.set_xlabel(meta['desc'], fontsize=7)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=7)

fig.suptitle('QM9 Target Property Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Graph size statistics

In [ ]:
N_STATS = 2000   # increase for more accurate stats (slower)
n_atoms_list, n_edges_list, n_angles_list, n_torsions_list = [], [], [], []

for i in range(N_STATS):
    d = ds[i]
    n_atoms_list.append(d.x.shape[0])
    n_edges_list.append(d.edge_index.shape[1])
    if hasattr(d, 'angle_attr'):
        n_angles_list.append(d.angle_attr.shape[0])
    if hasattr(d, 'torsion_attr'):
        n_torsions_list.append(d.torsion_attr.shape[0])

for label, lst in [
    ('Atoms per mol',    n_atoms_list),
    ('Edges per mol',    n_edges_list),
    ('Angles per mol',   n_angles_list),
    ('Torsions per mol', n_torsions_list),
]:
    arr = np.array(lst)
    print(f'{label:<20}  min={arr.min():<4}  max={arr.max():<5}  mean={arr.mean():.1f}  median={np.median(arr):.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(n_atoms_list, bins=range(1, max(n_atoms_list)+2), color='steelblue', edgecolor='white')
axes[0].set_title('Atoms per molecule')
axes[0].set_xlabel('# atoms')
axes[0].set_ylabel('Count')
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].hist(n_edges_list, bins=40, color='seagreen', edgecolor='white')
axes[1].set_title('Directed edges per molecule')
axes[1].set_xlabel('# edges')
axes[1].spines[['top', 'right']].set_visible(False)

fig.suptitle(f'Graph size distributions (first {N_STATS:,} molecules)')
plt.tight_layout()
plt.show()

## 8. Split & DataLoader smoke test

In [ ]:
train_idx, test_idx = make_splits(
    n_samples=len(ds),
    splits_dir=ROOT / 'data' / 'splits',
    dataset_name='qm9',
    train_ratio=0.80,
    seed=42,
)

print(f'Train : {len(train_idx):,}  ({len(train_idx)/len(ds)*100:.0f}%)')
print(f'Test  : {len(test_idx):,}  ({len(test_idx)/len(ds)*100:.0f}%)')
print(f'Overlap: {len(set(train_idx) & set(test_idx))} molecules')

In [ ]:
train_loader, test_loader = get_loaders(
    dataset=ds,
    train_idx=train_idx,
    test_idx=test_idx,
    batch_size=32,
)

train_batch = next(iter(train_loader))
test_batch  = next(iter(test_loader))

print('Train batch:', train_batch)
print()
print('Test  batch:', test_batch)
print()
print(f'Train batches/epoch : {len(train_loader)}')
print(f'Test  batches/epoch : {len(test_loader)}')

## 9. Atom-type distribution across dataset

In [ ]:
# one-hot columns 1–5 of x are [H, C, N, O, F]
from src.data.featurizer import ATOM_TYPES
from rdkit.Chem.Periodic import element as periodic

element_symbols = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F'}
counts = {sym: 0 for sym in element_symbols.values()}

N_CHECK = 5000
for i in range(N_CHECK):
    x = ds[i].x
    for j, (z, sym) in enumerate(element_symbols.items()):
        counts[sym] += int(x[:, j+1].sum().item())

total = sum(counts.values())
print(f'Atom composition across {N_CHECK:,} molecules:')
for sym, cnt in counts.items():
    print(f'  {sym}: {cnt:,}  ({cnt/total*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#aec6cf', '#3a7ebf', '#4caf50', '#e53935', '#9c27b0']
ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='white')
ax.set_title(f'Atom type counts (first {N_CHECK:,} molecules)')
ax.set_ylabel('# atoms')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()